# Nairobi station data pipeline

This notebook uses the four CSVs listed in `scripts/nairobi_pipeline/station_sources.json`. The exploratory source notebook is not required.

The workflow is **extract → clean → transform → analyse → load → train/validate → explicitly test**. Each stage is a Python function in `scripts/nairobi_pipeline/`.

**Important:** there are no observed flood labels in these weather files. The default experiment predicts whether the next target day's recorded rainfall exceeds a station-specific, early-training wet-day quantile. It is **not a flood probability, evacuation trigger, or operational weather forecast**. Upstream cleaning/imputation is undocumented, so even rainfall results are exploratory.

Read `scripts/nairobi_pipeline/README.md` for setup, assumptions and automation. Install its requirements in a research environment and select that Python kernel. Intel Macs without OpenMP can run `python -m scripts.nairobi_pipeline prepare-runtime` once.

Outputs are cleared for portable publication. See the pipeline README for recorded results.


In [ ]:
from pathlib import Path
import os
import sys
import json
import sqlite3
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "scripts/nairobi_pipeline/data.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook within the FinalYearProject repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.nairobi_pipeline.data import (
    Config, extract_data, clean_data, transform_data, analyse_data, mask_persistent_values, create_targets, split_data
)
from scripts.nairobi_pipeline.runner import run_pipeline, evaluate_test
config = Config()
display(pd.Series(config.validate(), name="Configuration"))

## 1. Extract only the selected sources

The explicit source manifest selects the four CSV files without executing a separate notebook. The metadata's degree/hemisphere strings become numeric coordinates; the metadata file uses a non-UTF-8 encoding. Filename IDs remain strings. The Doonholm `HKRE0`/metadata `HKJK` discrepancy is explicitly recorded, not silently treated as confirmed station identity.

In [ ]:
bundle = extract_data(config)
display(bundle["stations"])
display(pd.DataFrame(bundle["manifest"]["files"]))
display(bundle["raw"].groupby("station_id").size().rename("source_rows"))
display(bundle["raw"].head())

## 2. Clean and audit

Convert numeric text, reject invalid/future dates, mask impossible measurements and flag inconsistent temperature order. Collapse exact duplicate station-days; quarantine conflicting station-days. Insert missing **calendar** days with missing measurements, never zero rain. Original strings and source-row references remain in the raw snapshot.

The broad sanity bounds are configurable checks, not flood-warning thresholds. Extreme but plausible observations are not removed merely for being statistical outliers.

In [ ]:
cleaned, rejected, cleaning_audit = clean_data(bundle["raw"], bundle["stations"])
display(cleaning_audit)
display(cleaned[["time", "station_id", "tavg", "prcp", "calendar_inserted"]].dtypes)
display(rejected.head(10))
print("Raw records:", len(bundle["raw"]), "Daily calendar records:", len(cleaned))

## 3. Transform without looking forward

All weather predictors for target date **t** use only observations dated **t−1 or earlier**, separately by station. Calendar gaps remain missing. Rolling sums/means require complete windows; history counts describe missingness. Season and station location/identity are retained. A separate modelling view withholds fields from day 30 of an unchanged run onward; zero rainfall is exempt. This rule is causal and does not rewrite source values. No current-day rainfall or temperature is a model input.

In [ ]:
model_weather = mask_persistent_values(cleaned)
features = transform_data(model_weather)
display(features.head(12))
assert not {"prcp", "tavg", "tmin", "tmax", "wspd", "pres", "target"}.intersection(features.columns)
assert not features.duplicated(["station_id", "time"]).any()

## 4. Analyse quality before pooling

Coverage and suspicious runs are audited station by station. Constant non-zero rainfall runs and identical cross-station readings are flags to investigate, not automatic deletion rules. Descriptive statistics, correlations and monthly summaries are restricted to the training period. No station averaging or arbitrary weather bins are needed.

In [ ]:
analysis = analyse_data(cleaned, bundle["stations"], config.train_end)
display(analysis["station_quality"])
display(analysis["constant_runs"].sort_values("consecutive_days", ascending=False).head(15))
display(analysis["station_overlap"])
display(analysis["training_correlations"])
display(analysis["training_monthly_rainfall"].head(24))

## 5. Define the experimental target and chronological partitions

With no flood labels, the target is **recorded high rainfall**, strictly above each eligible station's 95th percentile of positive rainfall in the calibration window through 2019. The calibration window precedes every CV validation block. Stations need at least 365 calibration observations and 30 positive-rain calibration days and variation in positive amounts; these are configurable data-support checks, not evidence of statistical adequacy.

Training: through 2021. Validation: 2022–2023. Final test: 2024 onward. All stations on a given date stay in the same partition. Sparse or constant-calibration stations remain in the cleaned/audit outputs but may have no model target. Missing rain is not a negative label. At least three known prior rainfall days in the past week are required for model rows.

These are retrospective known-station experiments. They do not establish performance at unseen stations or original measurement availability at prediction time.

In [ ]:
labelled, eligibility = create_targets(model_weather, features, config)
modelling, partitions = split_data(labelled, config)
display(pd.DataFrame(eligibility))
display(pd.DataFrame([
    {"partition": name, "rows": len(frame), "first_date": frame.time.min(), "last_date": frame.time.max()}
    for name, frame in partitions.items()
]))
display(partitions["train"].groupby("station_id").target.agg(["count", "sum", "mean"]))

## 6. Automate loading and model comparison

`run_pipeline` executes all stages and writes versioned research CSVs and SQLite snapshots. It reuses a complete, hash-verified run when inputs, configuration, dependencies and code are unchanged. It does not write to the app's Supabase database.

Logistic Regression, Random Forest and XGBoost are compared with majority and previous-rain baselines. Every CV fit learns its own imputation/scaling/encoding. If a historical fold lacks a class, predeclared fixed settings are used instead, with the reason recorded; no random split is substituted. Validation average precision selects the model, with F1 as a tie-breaker. Decision threshold 0.5 is fixed in advance; the rainfall quantile and classifier decision threshold are different quantities.

The first run trains models; later unchanged runs load cached artefacts. This cell does not evaluate the final test set.

In [ ]:
run = run_pipeline(config)
print("Research run:", run)
display(pd.read_csv(run / "validation_comparison.csv"))
display(pd.read_csv(run / "validation_by_station.csv"))
selection = json.loads((run / "selection.json").read_text())
display(pd.Series(selection, name="Frozen selection"))
with sqlite3.connect(run / "research.sqlite") as connection:
    display(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", connection))

## 7. Explicit final test, after selection is frozen

Set the switch below to `True` only when ready to score the frozen test snapshot. Existing final-test outputs are displayed without refitting. Once inspected, that period is no longer untouched; do not tune against it and claim an independent test.

These scores concern the rainfall proxy only. Observed flood testing requires sourced, spatially matched positive **and negative** flood labels. A `mode="observed_labels"` path exists for that later dataset; it never turns missing event records into negatives.

In [ ]:
EVALUATE_FINAL_TEST = False
if EVALUATE_FINAL_TEST:
    evaluate_test(run)
if (run / "test_evaluation.json").exists():
    print("This frozen snapshot has already been evaluated. No model is being refitted here.")
    display(pd.read_csv(run / "test_metrics.csv"))
    display(pd.Series(json.loads((run / "test_evaluation.json").read_text())))
else:
    print("Final test has not been scored for this run.")

## 8. Automation and handover

From the repository root:

```sh
python -m scripts.nairobi_pipeline run
python -m scripts.nairobi_pipeline evaluate-test data/processed/nairobi_stations/<run-id>
python -m scripts.nairobi_pipeline schedule-template --hour 7 --minute 0
```

The generated macOS schedule is a template, not an installed recurring task. The one-command runner already automates all stages and detects changed inputs. It does not fetch new station data from a provider. Read `REPORT.md`, `TEST_REPORT.md` when present, source manifests, quality reports and frozen selections before interpreting scores.

**Remaining scientific boundary:** obtain the original station observations/imputation history and independently sourced flood outcomes before claiming or deploying a flood-risk model.